# Hello OpenImpala: AI-Ready Materials Design

**OpenImpala** is a high-performance framework that calculates effective transport
properties -- diffusivity, tortuosity, conductivity -- directly on 3D voxel images
of porous microstructures. Under the hood it uses finite differences on the voxel
grid (no mesh generation), parallelised via MPI through AMReX, with HYPRE for
linear solves.

With the new Python bindings, the entire C++/MPI/HYPRE stack is accessible from
a single `import openimpala`. You pass in a NumPy array and get physics back.

**In this tutorial we will:**

1. Generate a synthetic 3D gyroid microstructure in pure NumPy (no external data files).
2. Calculate volume fraction, percolation, and tortuosity with one function call each.
3. Run a parametric sweep of porosity vs. tortuosity -- the building block for
   autonomous, AI-driven inverse design of battery electrodes.

---

**Running this notebook:**

| Environment | Setup |
|---|---|
| **Inside the OpenImpala container** (recommended) | Everything is pre-installed. Just run the cells. |
| **Google Colab** | Run the setup cell below first -- it installs system dependencies and builds OpenImpala from source (~5 min). |
| **Local machine** | Follow the [build instructions](../CLAUDE.md) to install dependencies, then `pip install -e .` from the repo root. |

In [ ]:
#@title Colab Setup (skip if running locally or inside the container) { display-mode: "form" }
#
# This cell installs system-level dependencies (OpenMPI, HDF5, libtiff, HYPRE,
# AMReX) and builds OpenImpala + its Python bindings from source.
# Takes ~5 minutes on a standard Colab CPU runtime.
#
# If you are running inside the OpenImpala Apptainer/Docker container or a local
# dev environment where OpenImpala is already installed, skip this cell entirely.

import importlib, subprocess, sys, os

def _run(cmd, **kw):
    """Run a shell command, streaming output."""
    print(f">>> {cmd}")
    subprocess.check_call(cmd, shell=True, **kw)

if importlib.util.find_spec("openimpala") is None:
    print("openimpala not found -- installing from source...\n")

    # --- System packages (Ubuntu on Colab) ---
    _run("apt-get update -qq")
    _run("apt-get install -y -qq "
         "build-essential gfortran cmake "
         "libopenmpi-dev openmpi-bin "
         "libhdf5-openmpi-dev "
         "libtiff-dev "
         "python3-dev "
         "git wget > /dev/null 2>&1")

    # --- HYPRE (structured-grid solvers) ---
    HYPRE_VER = "2.31.0"
    if not os.path.exists("/usr/local/lib/libHYPRE.a"):
        _run(f"wget -q https://github.com/hypre-space/hypre/archive/v{HYPRE_VER}.tar.gz")
        _run(f"tar xzf v{HYPRE_VER}.tar.gz")
        _run(f"cd hypre-{HYPRE_VER}/src && "
             f"./configure --prefix=/usr/local --with-MPI --enable-shared=no "
             f"CC=mpicc CXX=mpicxx FC=mpif90 "
             f'CFLAGS="-O2" CXXFLAGS="-O2" '
             f"&& make -j$(nproc) && make install",
             stdout=subprocess.DEVNULL)
        print("HYPRE installed.")
    else:
        print("HYPRE already installed, skipping.")

    # --- AMReX (parallel mesh infrastructure) ---
    AMREX_VER = "25.03"
    if not os.path.exists("/usr/local/lib/cmake/AMReX"):
        _run(f"git clone --depth 1 --branch {AMREX_VER} "
             f"https://github.com/AMReX-Codes/amrex.git /tmp/amrex")
        _run("cmake -S /tmp/amrex -B /tmp/amrex/build "
             "-DCMAKE_INSTALL_PREFIX=/usr/local "
             "-DCMAKE_BUILD_TYPE=Release "
             "-DAMReX_MPI=ON -DAMReX_OMP=ON "
             "-DAMReX_SPACEDIM=3 "
             "-DAMReX_FORTRAN=ON "
             "-DAMReX_PARTICLES=OFF")
        _run("cmake --build /tmp/amrex/build -j$(nproc)",
             stdout=subprocess.DEVNULL)
        _run("cmake --install /tmp/amrex/build",
             stdout=subprocess.DEVNULL)
        print("AMReX installed.")
    else:
        print("AMReX already installed, skipping.")

    # --- pyAMReX (Python bindings for AMReX) ---
    _run(f"{sys.executable} -m pip install -q pyamrex pybind11")

    # --- Clone and build OpenImpala ---
    REPO = "https://github.com/BASE-Laboratory/OpenImpala.git"
    SRC = "/tmp/OpenImpala"
    if not os.path.exists(SRC):
        _run(f"git clone --depth 1 {REPO} {SRC}")

    BUILD = f"{SRC}/cmake_build"
    _run(f"cmake -S {SRC} -B {BUILD} "
         f"-DCMAKE_BUILD_TYPE=Release "
         f"-DOPENIMPALA_PYTHON=ON "
         f"-DBUILD_TESTING=OFF "
         f"-DCMAKE_PREFIX_PATH=/usr/local")
    _run(f"cmake --build {BUILD} -j$(nproc)", stdout=subprocess.DEVNULL)

    # Add the Python package to the path
    sys.path.insert(0, f"{SRC}/python")
    print("\nOpenImpala built and ready.")

else:
    print("openimpala is already installed.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import openimpala as oi

print(f"OpenImpala version: {oi.__version__}")

## 1. Generating a Synthetic Microstructure

Real battery electrodes are imaged with X-ray CT or FIB-SEM, producing 3D voxel
grids where each voxel is labelled by phase (pore, active material, binder, ...).

For this tutorial we skip the imaging step and generate a **gyroid** surface -- a
triply-periodic minimal surface widely used as a scaffold geometry in energy
materials. The gyroid equation is:

$$f(x,y,z) = \sin(x)\cos(y) + \sin(y)\cos(z) + \sin(z)\cos(x)$$

Thresholding this continuous field produces a binary voxel grid:
- **Phase 0** (solid matrix) where $f \leq t$
- **Phase 1** (void / pore space) where $f > t$

Adjusting the threshold $t$ controls the porosity.

In [ ]:
def generate_gyroid(size=64, porosity_target=0.40, frequency=4.0):
    """Generate a binary 3D gyroid microstructure.

    Parameters
    ----------
    size : int
        Number of voxels along each axis.
    porosity_target : float
        Desired volume fraction of the void phase (phase 1).
    frequency : float
        Number of gyroid unit-cell repetitions across the domain.

    Returns
    -------
    np.ndarray
        int32 array of shape (size, size, size) with phase IDs 0 and 1.
    """
    lin = np.linspace(0, 2 * np.pi * frequency, size, endpoint=False)
    x, y, z = np.meshgrid(lin, lin, lin, indexing="ij")

    # Gyroid level-set function
    field = np.sin(x) * np.cos(y) + np.sin(y) * np.cos(z) + np.sin(z) * np.cos(x)

    # Choose threshold so that the desired fraction of voxels are above it
    threshold = np.percentile(field, (1.0 - porosity_target) * 100.0)

    # Phase 0 = solid, Phase 1 = void (pore)
    binary = (field > threshold).astype(np.int32)
    return binary


N = 64
micro = generate_gyroid(size=N, porosity_target=0.40)
print(f"Microstructure shape: {micro.shape}")
print(f"Data type:           {micro.dtype}")
print(f"Unique phases:       {np.unique(micro)}")
print(f"Measured porosity:   {micro.mean():.4f}")

## 2. Visualising a 2D Slice

Let's look at the mid-plane cross-section of our generated structure.
Black = solid (phase 0), white = pore (phase 1).

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(micro[N // 2, :, :], cmap="gray", interpolation="nearest")
ax.set_title(f"Gyroid mid-plane slice (z = {N // 2})")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()
plt.show()

## 3. The OpenImpala Handoff

Now we pass our standard NumPy array directly into OpenImpala. Behind the scenes
the Python bindings will:

1. Convert the NumPy array into an AMReX `iMultiFab` (the parallel voxel container).
2. Initialise AMReX and HYPRE (MPI, domain decomposition, structured-grid solver).
3. Solve the steady-state diffusion equation on the voxel grid.
4. Return the results as plain Python objects.

All of this is wrapped in `oi.Session()`, which manages the C++ library lifecycle.

In [ ]:
with oi.Session():

    # --- Volume fraction ---
    # Phase 1 = pore space.  This should match our target porosity.
    vf = oi.volume_fraction(micro, phase=1)
    print(f"Porosity (phase 1 VF): {vf.fraction:.4f}  ({vf.fraction*100:.1f}%)")

    # --- Percolation check ---
    # Does the pore phase form a connected path from one face to the other?
    perc = oi.percolation_check(micro, phase=1, direction="z")
    print(f"Percolates in Z:       {perc.percolates}")
    print(f"Active volume fraction: {perc.active_volume_fraction:.4f}")

    # --- Tortuosity ---
    # Solve the diffusion equation and compute tortuosity.
    # Only meaningful if the phase percolates.
    if perc.percolates:
        result = oi.tortuosity(micro, phase=1, direction="z")
        print(f"Tortuosity (tau):      {result.tortuosity:.4f}")
        print(f"Solver converged:      {result.solver_converged}")
        print(f"Iterations:            {result.iterations}")
        print(f"Residual norm:         {result.residual_norm:.2e}")
    else:
        print("Phase does not percolate -- tortuosity is undefined.")

## 4. Parametric Sweep: Tortuosity vs. Porosity

Because OpenImpala is now accessible from Python, scripting parametric sweeps is
trivial. This is a foundational capability for **autonomous AI-driven inverse
design**: an optimisation loop can generate candidate microstructures, evaluate
them with OpenImpala, and iterate -- all without leaving Python.

Below we sweep over a range of target porosities, generate a gyroid for each,
and plot the resulting tortuosity curve. We use a smaller grid (48^3) to keep
the sweep fast.

In [ ]:
porosities = [0.35, 0.45, 0.55, 0.65]
results = []  # (porosity, tau) pairs

with oi.Session():
    for target in porosities:
        data = generate_gyroid(size=48, porosity_target=target)

        # Check percolation first
        perc = oi.percolation_check(data, phase=1, direction="z")
        if not perc.percolates:
            print(f"  porosity={target:.2f}  ->  does not percolate, skipping")
            continue

        # Solve for tortuosity
        res = oi.tortuosity(data, phase=1, direction="z")
        results.append((target, res.tortuosity))
        print(f"  porosity={target:.2f}  ->  tau={res.tortuosity:.4f}")

# --- Plot the sweep ---
if results:
    phi_vals, tau_vals = zip(*results)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(phi_vals, tau_vals, "o-", color="#1f77b4", linewidth=2, markersize=8)
    ax.set_xlabel("Porosity")
    ax.set_ylabel("Tortuosity")
    ax.set_title("Gyroid: Tortuosity vs. Porosity")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No percolating structures found -- try different porosity targets.")